In [1]:
import pandas as pd
import numpy as np
import joblib

pd.set_option("display.max_columns", None)

# Load the model created by occupancy_prediction.ipynb
model_bundle = joblib.load("occupancy_model.joblib")

model = model_bundle["model"]
features = model_bundle["features"]
USER_LAT = model_bundle["user_lat"]
USER_LON = model_bundle["user_lon"]

print("Occupancy model loaded successfully.")
print("Features:", features)
print("User location:", USER_LAT, USER_LON)


Occupancy model loaded successfully.
Features: ['price', 'avg_rating', 'distance_km']
User location: 28.6139 77.209


In [2]:
# Load the datasets needed for recommendation
occupancy = pd.read_csv("dataset/lot_occupancy_hourly.csv")
parking = pd.read_csv("dataset/parking_lots.csv")
reviews = pd.read_csv("dataset/reviews.csv")

# Create average rating for each parking lot
rating_df = (
    reviews.groupby("lot_id", as_index=False)
    .agg(
        avg_rating=("rating", "mean"),
        review_count=("rating", "count")
    )
)

# Build the parking-lot recommendation dataset
df = (
    occupancy
    .merge(
        parking[
            [
                "id", "name", "address", "latitude", "longitude",
                "price_per_hour", "total_slots", "vehicle_type", "status"
            ]
        ],
        left_on="lot_id",
        right_on="id",
        how="left"
    )
    .merge(rating_df, on="lot_id", how="left")
)

df["avg_rating"] = df["avg_rating"].fillna(reviews["rating"].mean())
df["price"] = df["price_per_hour"]

print("Merged shape:", df.shape)


Merged shape: (4575, 23)


In [3]:
# Calculate distance from the same user location stored with the occupancy model
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(
        np.radians, [lat1, lon1, lat2, lon2]
    )
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    return 2 * R * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine_km(
    USER_LAT,
    USER_LON,
    df["latitude"].values,
    df["longitude"].values
)


In [4]:
# One row per parking lot
lot_features = (
    df[
        [
            "lot_id", "name", "address",
            "price", "avg_rating", "distance_km"
        ]
    ]
    .drop_duplicates("lot_id")
    .copy()
)

# IMPORTANT: this prediction comes from occupancy_model.joblib
lot_features["predicted_occupancy_pct"] = np.clip(
    model.predict(lot_features[features]),
    0,
    100
)

lot_features["predicted_available_pct"] = (
    100 - lot_features["predicted_occupancy_pct"]
)

# Keep only approved parking lots
recommendations = lot_features.merge(
    parking[["id", "status", "vehicle_type", "total_slots"]],
    left_on="lot_id",
    right_on="id",
    how="left"
)

recommendations = recommendations[
    recommendations["status"] == "approved"
].copy()

recommendations.head()


,lot_id,name,address,price,avg_rating,distance_km,predicted_occupancy_pct,predicted_available_pct,id,status,vehicle_type,total_slots
0,08a20e54-8f66-49fb-89d7-59c930d42d9a,Rohini Parking A,"Rohini, Delhi",20.31,3.863636,21.624372,1.376127,98.623873,08a20e54-8f66-49fb-89d7-59c930d42d9a,approved,bike,77
1,14d550ff-098c-4a84-8f96-7f9776734a1b,Greater Kailash Parking E,"Greater Kailash, Delhi",23.07,4.000000,7.743913,1.426085,98.573915,14d550ff-098c-4a84-8f96-7f9776734a1b,approved,all,72
2,19c8358a-b82b-4ab4-b5a8-645fd399e303,Pitampura Parking D,"Pitampura, Delhi",24.66,3.909091,12.441327,2.847814,97.152186,19c8358a-b82b-4ab4-b5a8-645fd399e303,approved,all,37
3,237291de-6a6d-452f-ad82-2f915745a94a,Janakpuri Parking E,"Janakpuri, Delhi",41.59,3.888889,11.992302,2.300000,97.700000,237291de-6a6d-452f-ad82-2f915745a94a,approved,all,44
4,30c9d6df-abe8-45ab-842f-ab365e631bcd,Dwarka Parking A,"Dwarka, Delhi",40.07,3.872727,16.009274,2.317855,97.682145,30c9d6df-abe8-45ab-842f-ab365e631bcd,approved,car,43


In [5]:
#Recommendation score
'''
 **40%** availability
- **25%** rating
- **20%** distance
- **15%** price

Higher score = better recommendation.
'''



def minmax(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(1.0, index=series.index)
    return (series - mn) / (mx - mn)

recommendations["availability_score"] = minmax(
    100 - recommendations["predicted_occupancy_pct"]
)

recommendations["rating_score"] = (
    recommendations["avg_rating"] / 5
)

recommendations["distance_score"] = (
    1 - minmax(recommendations["distance_km"])
)

recommendations["price_score"] = (
    1 - minmax(recommendations["price"])
)

recommendations["recommendation_score"] = (
    0.40 * recommendations["availability_score"]
    + 0.25 * recommendations["rating_score"]
    + 0.20 * recommendations["distance_score"]
    + 0.15 * recommendations["price_score"]
) * 100

top_parking = (
    recommendations
    .sort_values("recommendation_score", ascending=False)
    [[
        "name", "address", "price", "avg_rating",
        "distance_km", "predicted_occupancy_pct",
        "predicted_available_pct", "recommendation_score",
        "vehicle_type", "total_slots"
    ]]
    .head(5)
)

top_parking


,name,address,price,avg_rating,distance_km,predicted_occupancy_pct,predicted_available_pct,recommendation_score,vehicle_type,total_slots
16,Saket Parking B,"Saket, Delhi",16.68,3.897436,9.513414,1.354027,98.645973,90.049048,ev,78
1,Greater Kailash Parking E,"Greater Kailash, Delhi",23.07,4.000000,7.743913,1.426085,98.573915,89.562375,all,72
6,Nehru Place Parking D,"Nehru Place, Delhi",24.00,3.767442,8.472644,1.829639,98.170361,84.168313,ev,56
24,Greater Kailash Parking B,"Greater Kailash, Delhi",26.96,4.054054,8.541457,2.141622,97.858378,82.000338,all,47
21,Lajpat Nagar Parking A,"Lajpat Nagar, Delhi",34.44,4.020408,6.694634,2.152742,97.847258,80.901661,car,47


In [6]:
# Best parking recommendation
best = top_parking.iloc[0]

print("Recommended Parking:")
print("Name:", best['name'])
print("Address:", best["address"])
print(f"Price: ₹{best['price']:.2f}/hour")
print(f"Rating: {best['avg_rating']:.2f}/5")
print(f"Distance: {best['distance_km']:.2f} km")
print(f"Predicted occupancy: {best['predicted_occupancy_pct']:.2f}%")
print(f"Predicted available: {best['predicted_available_pct']:.2f}%")
print(f"Recommendation score: {best['recommendation_score']:.2f}/100")


Recommended Parking:
Name: Saket Parking B
Address: Saket, Delhi
Price: ₹16.68/hour
Rating: 3.90/5
Distance: 9.51 km
Predicted occupancy: 1.35%
Predicted available: 98.65%
Recommendation score: 90.05/100
